# Aufgabe: Einführung in Python

### 1. Lesen Sie den Datensatz mit Pandas ein. 

In [2]:
# Importieren aller nötigen Pakete
import pandas as pd
import os
import numpy as np
from sklearn.preprocessing import LabelEncoder
from scipy import stats

In [ ]:
# navigieren zum Dateipfad in dem die Daten liegen
census = pd.read_csv(
    os.path.join("..", "..", "..", "data", "census.csv"), sep=",", header=0
)

# Einlesen der csv Datei mithilfe von pandas
# census = ...
census["sex"] = census["sex"].replace({" Male": 1, " Female": 0})
le = LabelEncoder() # nicht geeignet für Modellierung mit Regression oder kNN, da die Werte eine Ordnung suggerieren, die nicht existiert. Besser wäre dann One-Hot-Encoding.
census["education"] = le.fit_transform(census["education"])
census["workclass"] = le.fit_transform(census["workclass"])
census["occupation"] = le.fit_transform(census["occupation"])
census["native-country"] = le.fit_transform(census["native-country"])
census["marital-status"] = le.fit_transform(census["marital-status"])
census["relationship"] = le.fit_transform(census["relationship"])
census["race"] = le.fit_transform(census["race"])
census["target"] = le.fit_transform(census["target"])

#categorical_cols = census.select_dtypes(include=["object", "category"]).columns
#print("Kategorische Spalten:", categorical_cols.tolist())
#census = pd.get_dummies(census, columns=categorical_cols, drop_first=True)

census.head()
#print(census.dtypes)

# Denken Sie daran, dass nicht alle Spalten für diese Aufgabe benötigt werden:
# census.pop()

# Außerdem kann man keinen Durchschnittswert von kategorischen Features ermitteln.
# Einer der Wege, mit kategorischen Werten umzugehen:

# SEX: Male: 1, Female: 0


C:\Users\fried\AppData\Local\Temp\ipykernel_14416\2600723141.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  census["sex"] = census["sex"].replace({" Male": 1, " Female": 0})


,age,workclass,education,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,target
0,39,7,9,4,1,1,4,1,2174,0,40,39,0
1,50,6,9,2,4,0,4,1,0,0,13,39,0
2,38,4,11,0,6,1,4,1,0,0,40,39,0
3,53,4,1,2,6,0,2,1,0,0,40,39,0
4,28,4,9,2,10,5,2,0,0,0,40,5,0


### 2. Geben Sie eine Statistische Zusammenfassung der Daten aus:

Tip: mit Pandas möglich - in der [Dokumentation](https://pandas.pydata.org/docs/user_guide) kann es nachgelesen werden 

In [30]:
# Eine Übersicht über die verschiedenen Metriken wie Minimum, Maximum, Mittelwert
census.describe()
#census.dtypes

,age,capital-gain,capital-loss,hours-per-week
count,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1077.648844,87.303830,40.437456
std,13.640433,7385.292085,402.960219,12.347429
min,17.000000,0.000000,0.000000,1.000000
25%,28.000000,0.000000,0.000000,40.000000
50%,37.000000,0.000000,0.000000,40.000000
75%,48.000000,0.000000,0.000000,45.000000
max,90.000000,99999.000000,4356.000000,99.000000


### 3. Teilen Sie die Daten mittels Train-Test-Split von Sklearn in Trainings- und Testdaten

Tip: Machen Sie sich mit der Funktion und ihren Parametern vertraut: [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)

In [ ]:
from sklearn.model_selection import train_test_split

# Verwenden der Funktion train_test_split

# wenn Spaltenanordnung so bleibt:
#X = census.drop(["target"], axis=1)
#y = census["target"]
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

# Verwenden der Funktion train_test_split
#print("X Train: {}".format(X_train.shape))
#print("Y Train: {}".format(y_train.shape))
#print("X Test: {}".format(X_test.shape))
#print("Y Test: {}".format(y_test.shape))

# wenn Spalten erweitert werden, um kategorische Werte zu kodieren:
train, test = train_test_split(census, test_size=0.3)
print(train.shape)
print(test.shape)

(22792, 13)
(9769, 13)


### 4. Analysieren Sie mit Numpy, ob die Features in Test- und Trainingsdatensatz sich in den Mittelwerten unterscheiden.

In [ ]:


X_train = train.drop(["target"], axis=1) if "target" in train.columns else train
X_test = test.drop(["target"], axis=1) if "target" in test.columns else test

# nur numerische Features behalten
X_train = X_train.select_dtypes(include=[np.number])
X_test = X_test.select_dtypes(include=[np.number])
#print(X_train)

# In NumPy-Arrays konvertieren und sicherstellen, dass alles float ist
X_train_array = X_train.values.astype(float)
X_test_array = X_test.values.astype(float)
#print(X_train_array)

# Mittelwerte berechnen (für jede Spalte/Feature)
mean_train = np.mean(X_train_array, axis=0)
mean_test = np.mean(X_test_array, axis=0)
#print(mean_train)

# Mittelwerte der verschiedenen Features in Train / Test Data Frames betrachten
# Unterschied berechnen
differences = np.abs(mean_train - mean_test)

# Ausgabe
print("Unterschiede in Mittelwerten:")
for i, col in enumerate(X_train.columns):
    print(f"{col}: {differences[i]:.4f}")


# T-Test für jedes Feature
print("T-Test:")
for i, col in enumerate(X_train.columns):
    t_stat, p_value = stats.ttest_ind(X_train_array[:, i], X_test_array[:, i])
    print(f"{col}: t={t_stat:.4f}, p-value={p_value:.4f}")

Unterschiede in Mittelwerten:
age: 0.0384
workclass: 0.0096
education: 0.0174
marital-status: 0.0021
occupation: 0.0627
relationship: 0.0058
race: 0.0008
sex: 0.0074
capital-gain: 7.6427
capital-loss: 3.3972
hours-per-week: 0.0953
native-country: 0.1149
T-Test:
age: t=-0.2331, p-value=0.8157
workclass: t=-0.5464, p-value=0.5848
education: t=0.3725, p-value=0.7095
marital-status: t=0.1126, p-value=0.9103
occupation: t=-1.2265, p-value=0.2200
relationship: t=0.2974, p-value=0.7662
race: t=-0.0746, p-value=0.9406
sex: t=-1.2988, p-value=0.1940
capital-gain: t=-0.0856, p-value=0.9318
capital-loss: t=0.6971, p-value=0.4857
hours-per-week: t=-0.6381, p-value=0.5234
native-country: t=-1.2140, p-value=0.2248
